# FDR GPT-2 Fine-Tuning

Fine-tunes GPT-2 on the UCSB American Presidency Project FDR corpus to generate:
1. **Policy-grounded responses** — FDR-style text steered by policy domain (economy, labor, foreign policy, etc.)
2. **Iconic quotes** — short, punchy, quotable FDR-style lines

**Before running:**
1. Runtime > Change runtime type > **T4 GPU**
2. Upload your `fdr_ucsb/txt/` folder to Google Drive

**Sections:**
1. Setup & install
2. Load & inspect UCSB corpus
3. Policy domain classification
4. Extract iconic passages
5. Train / eval split
6. Tokenize
7. Fine-tune GPT-2
8. Generate: policy-grounded responses
9. Generate: iconic quotes
10. Batch generation for eval

## 1. Setup

In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, random, math, glob
import numpy as np
import torch
from collections import Counter
from transformers import (
    GPT2Tokenizer, GPT2LMHeadModel,
    TextDataset, DataCollatorForLanguageModeling,
    Trainer, TrainingArguments, pipeline
)

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 2. Load & Inspect UCSB Corpus

Reads individual `.txt` files from your Google Drive folder. Each file has a header block (Title/Date/Category/URL) followed by a `========` separator, then the speech body.

In [ ]:
# ── EDIT THIS PATH to match your Google Drive folder ──
CORPUS_DIR = '/content/drive/MyDrive/fdr_ucsb/txt'

txt_files = sorted(glob.glob(os.path.join(CORPUS_DIR, '*.txt')))
print(f'Found {len(txt_files)} text files in {CORPUS_DIR}')
if txt_files:
    print(f'First: {os.path.basename(txt_files[0])}')
    print(f'Last:  {os.path.basename(txt_files[-1])}')

In [ ]:
SEPARATOR = '=' * 72

def parse_ucsb_file(filepath):
    """Parse a single UCSB speech file into metadata + body text."""
    with open(filepath, 'r', encoding='utf-8') as f:
        raw = f.read()

    # Split on the ======== separator
    parts = raw.split(SEPARATOR, 1)
    header = parts[0] if len(parts) > 1 else ''
    body = parts[1].strip() if len(parts) > 1 else raw.strip()

    # Parse header fields (UCSB format: "Title:    value")
    meta = {}
    for line in header.split('\n'):
        line = line.strip()
        if line.startswith('Title:'):
            meta['title'] = line.split(':', 1)[1].strip()
        elif line.startswith('Date:'):
            meta['date'] = line.split(':', 1)[1].strip()
        elif line.startswith('Category:'):
            meta['category'] = line.split(':', 1)[1].strip()
        elif line.startswith('URL:'):
            meta['url'] = line.split(':', 1)[1].strip()

    return {'meta': meta, 'text': body, 'file': os.path.basename(filepath)}

# Parse all files
documents = []
skipped = 0
for fp in txt_files:
    doc = parse_ucsb_file(fp)
    # Skip very short documents (< 50 words)
    if len(doc['text'].split()) >= 50:
        documents.append(doc)
    else:
        skipped += 1

print(f'Parsed {len(documents)} documents ({skipped} skipped as too short)')
print(f'Sample title: {documents[0]["meta"].get("title", "?")}')
print(f'Sample date:  {documents[0]["meta"].get("date", "?")}')

In [ ]:
word_counts = [len(d['text'].split()) for d in documents]
print(f'Documents        : {len(documents)}')
print(f'Total words      : {sum(word_counts):,}')
print(f'Avg words/doc    : {int(np.mean(word_counts)):,}')
print(f'Median words/doc : {int(np.median(word_counts)):,}')
print(f'Min words/doc    : {min(word_counts)}')
print(f'Max words/doc    : {max(word_counts):,}')

cats = Counter(d['meta'].get('category', 'Unknown') for d in documents)
print('\nCategories:')
for cat, count in cats.most_common(20):
    print(f'  {count:4d}  {cat}')

# Date range
dates = [d['meta'].get('date', '') for d in documents if d['meta'].get('date')]
if dates:
    print(f'\nDate range: {dates[0]} — {dates[-1]}')

## 3. Policy Domain Classification

Tag each speech with a policy domain so the model learns to associate FDR's voice with specific topics. At generation time, you steer output by prompting with `[POLICY: Economy]` etc.

Domains are based on FDR's actual presidency:
- **Economy** — banking, depression, recovery, spending
- **Labor** — workers, wages, unions, employment
- **Foreign Policy** — war, allies, neutrality, defense
- **Social Welfare** — relief, Social Security, poverty, housing
- **Agriculture** — farms, crops, rural America
- **Government** — democracy, Constitution, reform, courts
- **Conservation** — land, resources, environment, CCC

In [ ]:
# ── Policy domain keyword classifier ──
POLICY_DOMAINS = {
    'Economy': [
        r'\bbank', r'\bdepress', r'\brecovery\b', r'\bbudget\b', r'\btax',
        r'\binflation\b', r'\bcredit\b', r'\bcurrency\b', r'\bwall street\b',
        r'\bspending\b', r'\bprosperity\b', r'\brecession\b', r'\bnra\b',
        r'\beconom', r'\bbusiness\b', r'\bindustr', r'\btariff',
    ],
    'Labor': [
        r'\bworker', r'\bwage', r'\bunion', r'\blabor\b', r'\bemploy',
        r'\bunemploy', r'\bjob[s ]', r'\bfair labor\b', r'\bworkingm',
        r'\bsweatshop', r'\bchild labor\b', r'\bcollective bargain',
    ],
    'Foreign Policy': [
        r'\bwar\b', r'\bpeace\b', r'\ball[yi]', r'\bnazi', r'\bhitler',
        r'\bjapan', r'\bgerman', r'\bneutral', r'\bdefense\b', r'\barmy\b',
        r'\bnavy\b', r'\bmilitar', r'\bweapon', r'\barsenal\b', r'\btyrann',
        r'\bdemocrac.*abroad', r'\bfascis', r'\baggress', r'\binvasion',
        r'\bunited nations\b', r'\blend.lease', r'\batlantic charter',
    ],
    'Social Welfare': [
        r'\brelief\b', r'\bsocial security', r'\bpoverty\b', r'\bhousing\b',
        r'\bwelfare\b', r'\bpension', r'\binsurance\b', r'\bhungr',
        r'\bhomeless', r'\bdestitu', r'\borphan', r'\bwpa\b', r'\bccc\b',
    ],
    'Agriculture': [
        r'\bfarm', r'\bcrop', r'\bagricultur', r'\brural\b', r'\bsoil\b',
        r'\bdrought\b', r'\bharvest', r'\bplant', r'\blivestock',
        r'\birrigat', r'\bdust bowl',
    ],
    'Government': [
        r'\bdemocracy\b', r'\bconstitution', r'\breform\b', r'\bcourt',
        r'\bcongress\b', r'\bsupreme court', r'\bvot', r'\belect',
        r'\brepublic\b', r'\bliberty\b', r'\bfreedom\b', r'\bright[s ]\b',
    ],
    'Conservation': [
        r'\bconserv', r'\bland\b', r'\bforest', r'\briver', r'\bwater\b',
        r'\bresource', r'\benviron', r'\bnational park', r'\bwildlife',
        r'\btva\b', r'\bdam\b', r'\berosion\b',
    ],
}

def classify_policy(text, top_n=2):
    """Return top N policy domains by keyword hit count."""
    text_lower = text.lower()
    scores = {}
    for domain, patterns in POLICY_DOMAINS.items():
        score = sum(len(re.findall(p, text_lower)) for p in patterns)
        if score > 0:
            scores[domain] = score
    if not scores:
        return ['General']
    ranked = sorted(scores, key=scores.get, reverse=True)
    return ranked[:top_n]

# Classify every document
for doc in documents:
    doc['domains'] = classify_policy(doc['text'])
    doc['primary_domain'] = doc['domains'][0]

# Show distribution
domain_counts = Counter(d['primary_domain'] for d in documents)
print('Policy domain distribution (primary):')
for domain, count in domain_counts.most_common():
    bar = '#' * (count // 2)
    print(f'  {domain:<20} {count:4d}  {bar}')

print(f'\nTotal classified: {len(documents)}')

## 4. Extract Iconic Passages

Pull out short, punchy paragraphs that have high rhetorical density — tricolons, moral framing, direct address. These get tagged `[ICONIC]` in training data so the model learns what "quotable FDR" sounds like, separate from long policy speeches.

In [ ]:
# ── Rhetorical pattern detectors ──
RHETORICAL_PATTERNS = [
    r'\bwe must\b',
    r'\blet us\b',
    r'\bI tell you\b',
    r'\bmy friends\b',
    r'\bthis generation\b',
    r'\bthis nation\b',
    r'\bwe have nothing to fear\b',
    r'\bthe only thing\b',
    r'\bI pledge\b',
    r'\bI ask\b',
    r'\bduty\b',
    r'\bdestiny\b',
    r'\bfreedom\b',
    r'\bliberty\b',
    r'\bjustice\b',
    r'\bfaith\b',
    r'\bcourage\b',
    r'\bsacrifice\b',
    r'\bnot .* but\b',           # "not X but Y" contrast
    r'\b\w+, \w+, and \w+\b',   # tricolon pattern
]

def rhetorical_score(paragraph):
    """Score a paragraph's rhetorical density (0-1)."""
    text_lower = paragraph.lower()
    hits = sum(1 for p in RHETORICAL_PATTERNS if re.search(p, text_lower))
    word_count = len(paragraph.split())
    if word_count == 0:
        return 0
    # Normalize: hits per 50 words, capped at 1.0
    return min(hits / max(word_count / 50, 1), 1.0)

def extract_iconic_passages(documents, min_words=15, max_words=120, min_score=0.3):
    """Extract short, rhetorically dense paragraphs from all documents."""
    iconic = []
    for doc in documents:
        paragraphs = doc['text'].split('\n\n')
        for para in paragraphs:
            para = para.strip()
            wc = len(para.split())
            if wc < min_words or wc > max_words:
                continue
            score = rhetorical_score(para)
            if score >= min_score:
                iconic.append({
                    'text': para,
                    'score': score,
                    'word_count': wc,
                    'source_title': doc['meta'].get('title', '?'),
                    'domain': doc['primary_domain'],
                })
    return sorted(iconic, key=lambda x: x['score'], reverse=True)

iconic_passages = extract_iconic_passages(documents)
print(f'Extracted {len(iconic_passages)} iconic passages')
print(f'\nTop 10 by rhetorical score:')
for i, p in enumerate(iconic_passages[:10]):
    preview = p['text'][:100].replace('\n', ' ')
    print(f'  {i+1}. [{p["score"]:.2f}] [{p["domain"]}] {preview}...')

## 5. Train / Eval Split

Shuffle by whole document, then format training data with:
- `[POLICY: Domain]` prefix on full speeches → teaches policy-grounded generation
- `[ICONIC]` prefix on extracted passages → teaches quotable, punchy output

Both tag types become sterable prompts at generation time.

In [ ]:
SEED       = 42
EVAL_RATIO = 0.10  # 90/10 split
EOS        = '<|endoftext|>'

random.seed(SEED)
shuffled = documents.copy()
random.shuffle(shuffled)

split_idx  = int(len(shuffled) * (1 - EVAL_RATIO))
train_docs = shuffled[:split_idx]
eval_docs  = shuffled[split_idx:]

# Also split iconic passages (keep them aligned with their source docs)
train_titles = {d['meta'].get('title') for d in train_docs}
train_iconic = [p for p in iconic_passages if p['source_title'] in train_titles]
eval_iconic  = [p for p in iconic_passages if p['source_title'] not in train_titles]

print(f'Train documents : {len(train_docs)}')
print(f'Eval documents  : {len(eval_docs)}')
print(f'Train words     : {sum(len(d["text"].split()) for d in train_docs):,}')
print(f'Eval words      : {sum(len(d["text"].split()) for d in eval_docs):,}')
print(f'Train iconic    : {len(train_iconic)} passages')
print(f'Eval iconic     : {len(eval_iconic)} passages')

In [ ]:
def format_training_text(docs, iconic):
    """Build training text with policy-tagged speeches + iconic-tagged passages."""
    parts = []

    # Full speeches with policy domain tags
    for doc in docs:
        domain = doc['primary_domain']
        tagged = f'[POLICY: {domain}] {doc["text"]}'
        parts.append(tagged)

    # Iconic passages with [ICONIC] tag (repeated to boost signal)
    for p in iconic:
        tagged = f'[ICONIC] {p["text"]}'
        parts.append(tagged)

    # Shuffle so iconic passages are interspersed, not clumped at the end
    random.shuffle(parts)
    return ('\n\n' + EOS + '\n\n').join(parts)

TRAIN_FILE = '/content/fdr_train.txt'
EVAL_FILE  = '/content/fdr_eval.txt'

with open(TRAIN_FILE, 'w', encoding='utf-8') as f:
    f.write(format_training_text(train_docs, train_iconic))

with open(EVAL_FILE, 'w', encoding='utf-8') as f:
    f.write(format_training_text(eval_docs, eval_iconic))

print(f'Train file: {os.path.getsize(TRAIN_FILE)/1e6:.1f} MB')
print(f'Eval file:  {os.path.getsize(EVAL_FILE)/1e6:.1f} MB')

# Sanity check: peek at first 500 chars of training data
with open(TRAIN_FILE, 'r') as f:
    print(f'\nFirst 500 chars of training data:')
    print(f.read(500))

## 6. Tokenize

GPT-2's tokenizer handles our `[POLICY: ...]` and `[ICONIC]` tags as regular tokens — the model will learn them as special prefixes during fine-tuning. No need to add special tokens.

In [ ]:
MODEL_NAME = 'gpt2'   # swap for 'gpt2-medium' if you want more capacity (slower)
BLOCK_SIZE = 512      # context window for training chunks

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

train_dataset = TextDataset(tokenizer=tokenizer, file_path=TRAIN_FILE, block_size=BLOCK_SIZE)
eval_dataset  = TextDataset(tokenizer=tokenizer, file_path=EVAL_FILE,  block_size=BLOCK_SIZE)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print(f'Train blocks : {len(train_dataset)}')
print(f'Eval blocks  : {len(eval_dataset)}')

# Show what a training example looks like after tokenization
sample = tokenizer.decode(train_dataset[0]['input_ids'][:80])
print(f'\nSample decoded block (first 80 tokens):')
print(sample)

## 7. Fine-Tune GPT-2

On a free Colab T4 (~15 GB VRAM), GPT-2 (124M params) trains in **20-40 min**. If you want better quality and have time, switch to `gpt2-medium` (355M) above.

In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive/fdr_gpt2_model'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_steps=50,
    report_to='none',
    learning_rate=5e-5,
    warmup_steps=100,
    weight_decay=0.01,
    fp16=True,
    prediction_loss_only=True,
)

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

In [ ]:
eval_results = trainer.evaluate()
perplexity = math.exp(eval_results['eval_loss'])
print(f"Eval loss  : {eval_results['eval_loss']:.4f}")
print(f"Perplexity : {perplexity:.2f}")
print()
print('Perplexity guide:')
print('  < 30   excellent — model has learned FDR\'s patterns well')
print('  30-60  good — usable for generation')
print('  60-100 fair — try more epochs or gpt2-medium')
print('  > 100  poor — check data quality or increase corpus size')

## 8. Generate: Policy-Grounded Responses

Prompt with `[POLICY: Domain]` to steer FDR into a specific policy lane. The model learned these tags during training, so it associates them with the right vocabulary and framing.

In [ ]:
generator = pipeline(
    'text-generation',
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device=0 if torch.cuda.is_available() else -1
)

def generate_policy(domain, opening, max_new_tokens=300, num_variants=2,
                    temperature=0.85, top_p=0.92):
    """Generate a policy-grounded FDR response in a specific domain."""
    prompt = f'[POLICY: {domain}] {opening}'
    outputs = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        num_return_sequences=num_variants,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2,
    )
    print(f'=== [POLICY: {domain}] ===')
    print(f'Prompt: "{opening}"\n')
    for i, out in enumerate(outputs):
        # Strip the [POLICY: ...] tag from output for readability
        text = out['generated_text']
        text = re.sub(r'^\[POLICY: [^\]]+\]\s*', '', text)
        print(f'--- Variant {i+1} ---')
        print(text)
        print()

def generate_iconic(opening='', max_new_tokens=80, num_variants=3,
                    temperature=0.95, top_p=0.9):
    """Generate short, punchy, quotable FDR-style lines."""
    prompt = f'[ICONIC] {opening}' if opening else '[ICONIC]'
    outputs = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        num_return_sequences=num_variants,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.3,
    )
    print(f'=== [ICONIC] ===')
    if opening:
        print(f'Prompt: "{opening}"\n')
    for i, out in enumerate(outputs):
        text = out['generated_text']
        text = re.sub(r'^\[ICONIC\]\s*', '', text)
        # Trim to first complete sentence for punchiness
        sentences = re.split(r'(?<=[.!?])\s+', text)
        # Take first 1-3 sentences
        quote = ' '.join(sentences[:3])
        print(f'  {i+1}. "{quote}"')
    print()

In [ ]:
# ── Policy-grounded examples on modern topics ──

# Economy: Congestion pricing
generate_policy('Economy',
    'My fellow Americans, the great arteries of our cities have become choked '
    'with the burden of the automobile, and it falls upon this government to')

# Labor: AI replacing jobs
generate_policy('Labor',
    'The machine has always been the servant of man, not his master. '
    'When new inventions displace the worker, it is the duty of government to')

# Foreign Policy: Global tech competition
generate_policy('Foreign Policy',
    'The democracies of the world face a new contest — not of armies '
    'but of industries and ingenuity. We must not allow the forces of')

# Social Welfare: Housing crisis
generate_policy('Social Welfare',
    'No family in America should want for shelter. The crisis of housing '
    'in our great cities is not merely an economic question but a moral one, and')

# Government: Social media and democracy
generate_policy('Government',
    'The instruments of communication, once the province of the printing press '
    'and the public square, have fallen into the hands of private enterprise, and')

## 9. Generate: Iconic Quotes

Short, punchy, quotable. The `[ICONIC]` tag tells the model to produce the kind of condensed, rhetorically powerful lines it saw in the extracted passages. Lower `max_new_tokens` + higher `temperature` = more creative, more concise.

In [ ]:
# ── Iconic quote generation ──

# Open-ended: let the model riff
generate_iconic()

# Seeded with FDR-style openings
generate_iconic('The only limit to our realization of tomorrow')
generate_iconic('We have always known that')
generate_iconic('Let it be said of this generation')
generate_iconic('The test of our progress is not')
generate_iconic('No democracy can long survive')
generate_iconic('My friends, the American people')

## 10. Batch Generation for Eval

Generate FDR responses to 50 modern headlines. These become the test cases for the Day 4 eval framework. Each response gets both a policy-grounded version and an iconic quote version, saved to a JSON file on Drive.

In [ ]:
import json

# ── 50 modern headlines mapped to FDR policy domains ──
MODERN_HEADLINES = [
    # Economy
    {"headline": "Fed Holds Interest Rates Steady Amid Inflation Fears", "domain": "Economy",
     "opening": "My friends, the management of our currency and credit is not a matter for bankers alone —"},
    {"headline": "Wall Street Hits Record High as Main Street Struggles", "domain": "Economy",
     "opening": "We cannot measure the prosperity of a nation by the ticker tape of Wall Street, for"},
    {"headline": "Crypto Exchange Collapses, Billions in Customer Funds Missing", "domain": "Economy",
     "opening": "Once again the speculators have played fast and loose with the savings of the common man, and"},
    {"headline": "Student Loan Debt Passes $1.7 Trillion", "domain": "Economy",
     "opening": "The youth of this nation, eager to learn and to serve, should not begin their lives shackled by"},
    {"headline": "Housing Prices Rise for 12th Consecutive Month", "domain": "Economy",
     "opening": "A home is not a speculation — it is a right. When the cost of shelter exceeds the reach of the working family,"},
    # Labor
    {"headline": "AI Chatbot Replaces 700 Customer Service Workers", "domain": "Labor",
     "opening": "The machine has always been the servant of man, not his master. When new inventions displace the worker,"},
    {"headline": "Amazon Warehouse Workers Vote to Unionize", "domain": "Labor",
     "opening": "The right of workers to organize is not a privilege granted by their employers — it is"},
    {"headline": "Gig Workers Denied Benefits in Court Ruling", "domain": "Labor",
     "opening": "No man or woman who gives an honest day's work should be denied the basic protections of"},
    {"headline": "Tech Layoffs Hit 100,000 This Quarter", "domain": "Labor",
     "opening": "When great industries cast aside their workers in the name of efficiency, it falls upon"},
    {"headline": "Minimum Wage Hasn't Kept Pace with Inflation Since 2009", "domain": "Labor",
     "opening": "A wage that cannot sustain a family is not a wage — it is an injustice, and"},
    # Foreign Policy
    {"headline": "China Surpasses US in AI Patent Filings", "domain": "Foreign Policy",
     "opening": "The democracies of the world face a new contest — not of armies but of industries and ingenuity, and"},
    {"headline": "NATO Allies Debate Defense Spending Amid Rising Tensions", "domain": "Foreign Policy",
     "opening": "The peace of the world cannot rest upon the goodwill of aggressors. Our allies and our own people must"},
    {"headline": "Russia Expands Military Operations in Eastern Europe", "domain": "Foreign Policy",
     "opening": "The forces of tyranny do not respect treaties or borders. We have seen this before, and"},
    {"headline": "UN Climate Summit Ends Without Binding Agreement", "domain": "Foreign Policy",
     "opening": "The nations of the world have gathered and spoken, yet have not acted. If we cannot find the courage to"},
    {"headline": "Global Chip Shortage Threatens National Security", "domain": "Foreign Policy",
     "opening": "A nation that cannot manufacture the instruments of its own defense is a nation at the mercy of others, and"},
    # Social Welfare
    {"headline": "Homelessness Rises 12% in Major US Cities", "domain": "Social Welfare",
     "opening": "In the richest nation the world has ever known, no citizen should sleep upon the streets. The measure of our civilization is"},
    {"headline": "Mental Health Crisis Among Teens Worsens", "domain": "Social Welfare",
     "opening": "The well-being of our children is the first duty of this republic. When the young suffer in silence,"},
    {"headline": "Rural Hospitals Continue to Close Across America", "domain": "Social Welfare",
     "opening": "The farmer and the small-town family deserve the same care as the city dweller. No American should die because"},
    {"headline": "Food Banks Report Record Demand This Holiday Season", "domain": "Social Welfare",
     "opening": "In a land of abundance, hunger is not a failure of supply — it is a failure of conscience, and"},
    {"headline": "Opioid Deaths Hit New Record in 2025", "domain": "Social Welfare",
     "opening": "A plague has settled upon our communities, not of foreign origin but of our own making, and"},
    # Government / Democracy
    {"headline": "Linguistics Professor Calls Colleague 'Mansplainer' in AI Translation Clash", "domain": "Government",
     "opening": "The pursuit of knowledge demands not insult but inquiry. When the scholars of a great nation descend to"},
    {"headline": "Voter Turnout Hits Historic Low in Midterm Elections", "domain": "Government",
     "opening": "Democracy does not die in dramatic fashion — it withers when the citizen turns away from the ballot box, and"},
    {"headline": "Social Media Algorithm Blamed for Political Polarization", "domain": "Government",
     "opening": "The instruments of communication, once the province of the public square, have been captured by machines that"},
    {"headline": "Supreme Court Limits Federal Agency Power in Landmark Ruling", "domain": "Government",
     "opening": "The Constitution is a living charter, not a straitjacket. When the courts constrain the ability of the people's government to"},
    {"headline": "Trust in Government Hits All-Time Low in New Poll", "domain": "Government",
     "opening": "The faith of the people in their government is not a luxury — it is the foundation, and when that faith erodes,"},
    # Agriculture / Environment
    {"headline": "Drought Devastates Midwest Corn Crop", "domain": "Agriculture",
     "opening": "The land does not lie. When the rains fail and the soil cracks, it is not merely the farmer who suffers — it is"},
    {"headline": "Family Farms Disappear as Corporate Agriculture Expands", "domain": "Agriculture",
     "opening": "The family farm is not a relic of the past — it is the backbone of this republic, and"},
    {"headline": "Wildfire Season Breaks Records for Third Straight Year", "domain": "Conservation",
     "opening": "The forests and the rivers and the air we breathe are held in trust for future generations, and"},
    {"headline": "Clean Water Crisis Affects 2 Million Americans", "domain": "Conservation",
     "opening": "Water is life itself. When the taps of an American city run with poison, it is not merely a failure of pipes — it is"},
    {"headline": "Flooding Displaces Thousands Along Mississippi River", "domain": "Conservation",
     "opening": "The great rivers of this continent have always been both our blessing and our challenge. We must"},
    # Tech & Modern Issues (mapped to closest FDR domain)
    {"headline": "Hospital Hacked, Patient Data Held for Ransom", "domain": "Government",
     "opening": "The safety of the citizen — whether in person or in the records that bear their name — is a sacred trust, and"},
    {"headline": "Electric Vehicle Sales Surge Past 50% Market Share", "domain": "Economy",
     "opening": "The American genius for industry has always risen to meet the challenge of the age. Today that challenge is"},
    {"headline": "Teachers Strike Over Pay and Class Sizes", "domain": "Labor",
     "opening": "The teacher stands at the foundation of our democracy. When we fail to honor their service with fair compensation,"},
    {"headline": "Prescription Drug Prices Rise 30% in Five Years", "domain": "Social Welfare",
     "opening": "No American should choose between their medicine and their meals. The profiteering of the pharmaceutical industry is"},
    {"headline": "Infrastructure Bill Stalls in Congress", "domain": "Economy",
     "opening": "The bridges and roads and rails of this nation are the arteries of commerce and connection, and when they crumble,"},
    {"headline": "Immigration Debate Paralyzes Congress", "domain": "Government",
     "opening": "This nation was built by the hands and the hopes of those who came to these shores seeking a better life, and"},
    {"headline": "Billionaire Space Race Criticized Amid Poverty on Earth", "domain": "Economy",
     "opening": "The ingenuity of the few must not blind us to the needs of the many. While some race toward the stars,"},
    {"headline": "Public Schools Face Massive Teacher Shortage", "domain": "Social Welfare",
     "opening": "The schoolhouse is the workshop of democracy. When we cannot staff it, we are building our future on"},
    {"headline": "College Enrollment Drops for Fifth Year Running", "domain": "Social Welfare",
     "opening": "A nation that closes the doors of learning to its young people is a nation that has lost faith in"},
    {"headline": "Police Reform Bill Dies in Senate", "domain": "Government",
     "opening": "Justice delayed is justice denied, and justice denied is the seed of unrest. When the people cry out for reform and"},
    {"headline": "Insulin Costs Force Diabetics to Ration Medication", "domain": "Social Welfare",
     "opening": "That any citizen of this great republic should go without the medicine that sustains their life is"},
    {"headline": "Small Business Closures Hit Pandemic-Era Highs", "domain": "Economy",
     "opening": "The small business — the shop on the corner, the family enterprise — is the true engine of American prosperity, and"},
    {"headline": "Rent Control Debate Heats Up in Major Cities", "domain": "Social Welfare",
     "opening": "Shelter is not a commodity to be traded on an exchange. It is a necessity, and when the cost of a roof exceeds"},
    {"headline": "Military Recruitment Falls Short for Second Year", "domain": "Foreign Policy",
     "opening": "The defense of this nation has always rested upon the willingness of its citizens to serve, and when that willingness falters,"},
    {"headline": "Climate Refugees Expected to Reach 200 Million by 2050", "domain": "Conservation",
     "opening": "The displacement of millions by the changing climate is not a distant prophecy — it is happening now, and"},
    {"headline": "Big Tech Antitrust Case Goes to Trial", "domain": "Government",
     "opening": "When any industry grows so large that it controls the marketplace and silences competition, it is the duty of the people's government to"},
    {"headline": "National Debt Passes $35 Trillion", "domain": "Economy",
     "opening": "A great nation must be solvent — not merely in its treasury but in its obligations to the next generation, and"},
    {"headline": "Pandemic Preparedness Funding Cut by 40%", "domain": "Social Welfare",
     "opening": "We have learned at terrible cost that preparedness is not an expense — it is an investment. To cut the defenses against"},
    {"headline": "Trade War Escalates with New Tariffs on Both Sides", "domain": "Foreign Policy",
     "opening": "Trade among nations should be a bridge, not a battleground. When tariffs become weapons rather than tools,"},
    {"headline": "Youth Voter Registration Surges After Campus Organizing", "domain": "Government",
     "opening": "The young people of this nation have heard the call of their generation. When the youth march to the ballot box,"},
]

print(f'Prepared {len(MODERN_HEADLINES)} headlines for batch generation')

In [ ]:
# ── Batch generate all 50 responses ──
eval_responses = []

for i, h in enumerate(MODERN_HEADLINES):
    print(f'[{i+1}/{len(MODERN_HEADLINES)}] {h["headline"][:60]}...', end=' ', flush=True)

    # Policy-grounded response
    prompt_policy = f'[POLICY: {h["domain"]}] {h["opening"]}'
    policy_out = generator(
        prompt_policy,
        max_new_tokens=250,
        num_return_sequences=1,
        temperature=0.85,
        top_p=0.92,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2,
    )
    policy_text = re.sub(r'^\[POLICY: [^\]]+\]\s*', '', policy_out[0]['generated_text'])

    # Iconic quote
    prompt_iconic = f'[ICONIC] {h["opening"].split(".")[0]}.'
    iconic_out = generator(
        prompt_iconic,
        max_new_tokens=80,
        num_return_sequences=1,
        temperature=0.95,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.3,
    )
    iconic_text = re.sub(r'^\[ICONIC\]\s*', '', iconic_out[0]['generated_text'])
    # Trim iconic to first 2-3 sentences
    sentences = re.split(r'(?<=[.!?])\s+', iconic_text)
    iconic_text = ' '.join(sentences[:3])

    eval_responses.append({
        'id': f'headline_{i+1:02d}',
        'headline': h['headline'],
        'domain': h['domain'],
        'opening_prompt': h['opening'],
        'policy_response': policy_text,
        'iconic_quote': iconic_text,
    })
    print('done')

# Save to Drive for use in eval framework
EVAL_OUTPUT = '/content/drive/MyDrive/fdr_eval_responses.json'
with open(EVAL_OUTPUT, 'w') as f:
    json.dump(eval_responses, f, indent=2)

print(f'\nSaved {len(eval_responses)} response pairs to {EVAL_OUTPUT}')
print('Ready for Day 4 eval framework!')

In [ ]:
# ── Preview a few responses ──
print('=' * 70)
print('  SAMPLE GENERATED RESPONSES')
print('=' * 70)

for r in eval_responses[:5]:
    print(f'\nHEADLINE: {r["headline"]}')
    print(f'DOMAIN:   {r["domain"]}')
    print(f'\nPOLICY RESPONSE:')
    print(f'  {r["policy_response"][:300]}...')
    print(f'\nICONIC QUOTE:')
    print(f'  "{r["iconic_quote"]}"')
    print('-' * 70)